## 1. Environment Setup and Library Imports

In [3]:
import os, sys, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if "__file__" not in globals() else Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

# Import local custom modules
from src.pkldataset import PKLDataset
from src.helpers import set_seed, get_model, train_model, eval_model

# Hardware configuration and loss function
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

# 3 seeds for quick verification before running all 10
seeds = [101, 202, 303]

# Sheet thickness and modality configurations
# Using T197 as the baseline based on the directory structure
source_thickness = "T197"
target_thickness = "T197"

modalities_config = [
    {
        "name": "RP_alone",
        "label": "Only Force (RP)",
        "hc_folder": str(DATA_DIR / "HC" / source_thickness / "RP"),
        "dc_folder": str(DATA_DIR / "DC" / target_thickness / "RP"),
        "channels": 1,
        "mode": "single"
    },
    {
        "name": "AZ_alone",
        "label": "Only Acceleration (AZ)",
        "hc_folder": str(DATA_DIR / "HC" / source_thickness / "AZ"),
        "dc_folder": str(DATA_DIR / "DC" / target_thickness / "AZ"),
        "channels": 1,
        "mode": "single"
    },
    {
        "name": "Fusion",
        "label": "Multimodal fusion (RP + AZ)",
        "hc_folder": str(DATA_DIR / "HC" / source_thickness / "RP"),
        "dc_folder": str(DATA_DIR / "DC" / target_thickness / "RP"),
        "channels": 2,
        "mode": "fusion"
    }
]

# 5. Verify dataset path existence
print("\nVerifying folder paths...")
for m in modalities_config:
    hc_ok = os.path.exists(m["hc_folder"])
    dc_ok = os.path.exists(m["dc_folder"])
    print(f"[{m['name']}] HC ({m['hc_folder']}): {'FOUND' if hc_ok else 'MISSING'} | DC ({m['dc_folder']}): {'FOUND' if dc_ok else 'MISSING'}")


Verifying folder paths...
[RP_alone] HC (c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T197\RP): FOUND | DC (c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\DC\T197\RP): FOUND
[AZ_alone] HC (c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T197\AZ): FOUND | DC (c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\DC\T197\AZ): FOUND
[Fusion] HC (c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T197\RP): FOUND | DC (c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\DC\T197\RP): FOUND


## 2. Configuration (RP Force Sensor)

In [ ]:
import sys
from pathlib import Path
import numpy as np
from torch.utils.data import Subset

# -------------------------------------------------------------------------
# Path resolution
# -------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent if "__file__" not in globals() else Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pkldataset import PKLDataset, class_to_onehot

# -------------------------------------------------------------------------
# Stratified Sampling Helper (Supports both PKLDataset and Subset objects)
# -------------------------------------------------------------------------
def get_balanced_transfer_subset(dataset, total_samples, seed=42):
    """
    Extracts an exactly class-balanced subset from a PKLDataset or Subset instance.
    For example: 20 samples -> 2 samples per wear class (R05..R50).
    """
    np.random.seed(seed)
    num_classes = 10
    samples_per_class = total_samples // num_classes
    
    # Handle both full PKLDataset instances and Subset splits
    if isinstance(dataset, Subset):
        base_dataset = dataset.dataset
        available_indices = dataset.indices
    else:
        base_dataset = dataset
        available_indices = list(range(len(dataset.files)))
    
    # Map subset indices by wear class (R05 to R50)
    class_indices = {i: [] for i in range(num_classes)}
    for idx in available_indices:
        filename = base_dataset.files[idx]
        class_label = filename.split('_')[0]
        c_idx = int(np.argmax(class_to_onehot[class_label]))
        class_indices[c_idx].append(idx)
        
    selected_indices = []
    for c_idx, indices in class_indices.items():
        chosen = np.random.choice(indices, size=samples_per_class, replace=False)
        selected_indices.extend(chosen)
        
    return Subset(base_dataset, selected_indices)

# -------------------------------------------------------------------------
# Dataset Paths & Configurations (Force Sensor - RP)
# -------------------------------------------------------------------------
# Primary pretraining path: HC source domain (T197 with force sensor RP)
train_path_1 = str(DATA_DIR / "HC" / "T197" / "RP")

# Target domain base path for fine-tuning: DC base folder
target_train_path = str(DATA_DIR / "DC" / "T197" / "RP")

# Transfer learning sample sizes (simulating train_20, train_50, ..., train_500)
transfer_sample_sizes = [20, 50, 100, 200, 300, 400, 500]
transfer_sets = [f"train_{size}" for size in transfer_sample_sizes]

# Validation sets across all DC sheet thicknesses (held-out evaluation sets)
val_paths = [
    str(DATA_DIR / "DC" / "T185" / "RP"),
    str(DATA_DIR / "DC" / "T188" / "RP"),
    str(DATA_DIR / "DC" / "T191" / "RP"),
    str(DATA_DIR / "DC" / "T194" / "RP"),   # -------------------------------------------------------------------------
# Preload validation datasets into RAM to avoid redundant disk I/O
# -------------------------------------------------------------------------
print("Preloading validation datasets into RAM...")
cached_val_tensors = {}

for vp in val_paths:
    # Use isolated test split for T197 to eliminate data leakage
    if "T197" in vp:
        temp_ds = test_ds_t197
    else:
        temp_ds = PKLDataset(folder_path=vp, modality="single")
        
    temp_loader = DataLoader(temp_ds, batch_size=256, shuffle=False)
    
    all_inputs, all_targets = [], []
    for x, y in temp_loader:
        all_inputs.append(x)
        all_targets.append(y)
        
    cached_val_tensors[vp] = {
        "inputs": torch.cat(all_inputs, dim=0),
        "targets": torch.cat(all_targets, dim=0)
    }

print("Validation datasets successfully preloaded into RAM.")
    str(DATA_DIR / "DC" / "T197" / "RP"),
]

# Random seeds for reproducibility across runs
seeds = [101, 202, 303, 404, 505, 606, 707, 808, 909, 1001]

# -------------------------------------------------------------------------
# Strict 70/30 Split on Target Domain (DC / T197) to eliminate data leakage
# -------------------------------------------------------------------------
# Reserve 70% of T197 as the pool for transfer learning subsets (up to train_500)
# Reserve 30% strictly as an isolated test set for evaluating the T197 column
train_pool_dc, test_ds_t197 = PKLDataset.split_dataset(
    folder_path=target_train_path,
    modality="single",
    train_ratio=0.7,
    random_seed=42
)

# Prepare results container matching the reference format: results[t][vp] -> []
results = {t: {vp: [] for vp in val_paths} for t in transfer_sets}

print(f"HC Source Pretraining Path: {train_path_1}")
print(f"DC Target Fine-Tuning Path: {target_train_path}")
print(f"DC Transfer Sets to Evaluate: {transfer_sets}")
print(f"DC Validation Thickness Paths: {val_paths}")
print(f"DC T197 Train Pool Size (70%): {len(train_pool_dc)} samples")
print(f"DC T197 Held-Out Test Set Size (30%): {len(test_ds_t197)} samples")
print(f"Total Random Seeds: {len(seeds)}")

HC Source Pretraining Path: c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T197\RP
DC Target Fine-Tuning Path: c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\DC\T197\RP
DC Transfer Sets to Evaluate: ['train_20', 'train_50', 'train_100', 'train_200', 'train_300', 'train_400', 'train_500']
DC Validation Thickness Paths: ['c:\\Users\\antho\\OneDrive\\Escritorio\\Erstsemester_Anthony\\Bachelorthesis\\Training_Daten_Stanzen\\Data\\Data\\data\\DC\\T185\\RP', 'c:\\Users\\antho\\OneDrive\\Escritorio\\Erstsemester_Anthony\\Bachelorthesis\\Training_Daten_Stanzen\\Data\\Data\\data\\DC\\T188\\RP', 'c:\\Users\\antho\\OneDrive\\Escritorio\\Erstsemester_Anthony\\Bachelorthesis\\Training_Daten_Stanzen\\Data\\Data\\data\\DC\\T191\\RP', 'c:\\Users\\antho\\OneDrive\\Escritorio\\Erstsemester_Anthony\\Bachelorthesis\\Training_Daten_Stanzen\\Data\\Data\\data\\DC\\T194\\RP', 'c:\\Use

## 3. Validation Preloading - RP

In [4]:
# -------------------------------------------------------------------------
# Preload validation datasets into RAM to avoid redundant disk I/O
# -------------------------------------------------------------------------
print("Preloading validation datasets into RAM...")
cached_val_tensors = {}

for vp in val_paths:
    temp_ds = PKLDataset(folder_path=vp, modality="single")
    temp_loader = DataLoader(temp_ds, batch_size=256, shuffle=False)
    
    all_inputs, all_targets = [], []
    for x, y in temp_loader:
        all_inputs.append(x)
        all_targets.append(y)
        
    cached_val_tensors[vp] = {
        "inputs": torch.cat(all_inputs, dim=0),
        "targets": torch.cat(all_targets, dim=0)
    }

print("Validation datasets successfully preloaded into RAM.")

Preloading validation datasets into RAM...
Validation datasets successfully preloaded into RAM.


## 4. Training and Evaluation - RP

In [5]:
# -------------------------------------------------------------------------
# Evaluation helper using in-memory cached tensors
# -------------------------------------------------------------------------
def fast_eval(model, inputs_tensor, targets_tensor, batch_size=256, device="cuda"):
    model.eval()
    correct = 0
    total = targets_tensor.size(0)
    with torch.no_grad():
        for i in range(0, total, batch_size):
            x = inputs_tensor[i:i + batch_size].to(device)
            y = targets_tensor[i:i + batch_size].to(device)
            preds = model(x).argmax(dim=1)
            targets_idx = y.argmax(dim=1)
            correct += (preds == targets_idx).sum().item()
    return 100.0 * correct / total

# -------------------------------------------------------------------------
# Phase 1 & 2: Pretraining on HC and Fine-Tuning across Transfer Sets
# -------------------------------------------------------------------------

for seed in seeds:
    print(f"\n==========================================")
    print(f">>> Running Experiment for Seed: {seed}")
    print(f"==========================================")
    set_seed(seed)

    # ---------------------------------------------------------------------
    # Phase 1: Pretraining on HC source dataset
    # ---------------------------------------------------------------------
    print("--> Phase 1: Pretraining on HC (T197)...")
    train_ds_hc, val_ds_hc = PKLDataset.split_dataset(
        folder_path=train_path_1,
        modality="single",
        train_ratio=0.8,
        random_seed=seed
    )
    train_loader_hc = DataLoader(train_ds_hc, batch_size=64, shuffle=True)

    # Initialize model with 1 input channel (RP force sensor)
    base_model = get_model(input_channels=1).to(device)
    opt_hc = optim.Adam(base_model.parameters(), lr=1e-3, weight_decay=1e-5)
    sch_hc = optim.lr_scheduler.StepLR(opt_hc, step_size=50, gamma=0.1)

    base_model = train_model(
        model=base_model,
        loader=train_loader_hc,
        criterion=criterion,
        optimizer=opt_hc,
        scheduler=sch_hc,
        num_epochs=10,
        device=device
    )

    # Save frozen weights from HC source domain
    pretrained_state = base_model.state_dict()

    # ---------------------------------------------------------------------
    # Phase 2: Transfer Learning (Fine-Tuning on DC subsets)
    # ---------------------------------------------------------------------
    for t_name, n_samples in zip(transfer_sets, transfer_sample_sizes):
        print(f"\n--- Fine-tuning on Transfer Set: {t_name} ({n_samples} samples) ---")
        
        # Extract class-balanced subset strictly from the 70% training pool
        transfer_subset = get_balanced_transfer_subset(
            dataset=train_pool_dc,
            total_samples=n_samples,
            seed=seed
        )
        batch_sz = min(32, n_samples)
        loader_t = DataLoader(transfer_subset, batch_size=batch_sz, shuffle=True)

        # Initialize new model and load HC pretrained weights
        tl_model = get_model(input_channels=1).to(device)
        tl_model.load_state_dict(pretrained_state)

        opt_dc = optim.Adam(tl_model.parameters(), lr=1e-3, weight_decay=1e-5)
        sch_dc = optim.lr_scheduler.StepLR(opt_dc, step_size=25, gamma=0.1)

        # Fine-tune on DC samples for 100 epochs
        tl_model = train_model(
            model=tl_model,
            loader=loader_t,
            criterion=criterion,
            optimizer=opt_dc,
            scheduler=sch_dc,
            num_epochs=100,
            device=device
        )

        # -----------------------------------------------------------------
        # Fast Evaluation across all held-out DC sheet thicknesses from RAM
        # -----------------------------------------------------------------
        for vp in val_paths:
            acc = fast_eval(
                model=tl_model,
                inputs_tensor=cached_val_tensors[vp]["inputs"],
                targets_tensor=cached_val_tensors[vp]["targets"],
                device=device
            )
            results[t_name][vp].append(acc)
            print(f"[Seed {seed}] {t_name} -> {vp}: {acc:.2f}%")


>>> Running Experiment for Seed: 101
--> Phase 1: Pretraining on HC (T197)...
Epoch 1/10, Loss: 0.7213
Epoch 2/10, Loss: 0.1686
Epoch 3/10, Loss: 0.1427
Epoch 4/10, Loss: 0.0977
Epoch 5/10, Loss: 0.1027
Epoch 6/10, Loss: 0.0927
Epoch 7/10, Loss: 0.0978
Epoch 8/10, Loss: 0.0812
Epoch 9/10, Loss: 0.0676
Epoch 10/10, Loss: 0.0649

--- Fine-tuning on Transfer Set: train_20 (20 samples) ---
Epoch 1/100, Loss: 9.1351
Epoch 2/100, Loss: 5.3910
Epoch 3/100, Loss: 2.3113
Epoch 4/100, Loss: 1.0944
Epoch 5/100, Loss: 0.9292
Epoch 6/100, Loss: 0.5726
Epoch 7/100, Loss: 0.5276
Epoch 8/100, Loss: 0.4630
Epoch 9/100, Loss: 0.6164
Epoch 10/100, Loss: 0.8364
Epoch 11/100, Loss: 0.2108
Epoch 12/100, Loss: 0.6216
Epoch 13/100, Loss: 0.3093
Epoch 14/100, Loss: 0.3706
Epoch 15/100, Loss: 0.6670
Epoch 16/100, Loss: 0.4238
Epoch 17/100, Loss: 0.4478
Epoch 18/100, Loss: 0.3131
Epoch 19/100, Loss: 0.1334
Epoch 20/100, Loss: 0.2541
Epoch 21/100, Loss: 0.2570
Epoch 22/100, Loss: 0.1871
Epoch 23/100, Loss: 0.472

## 5. Summary of Results - RP

In [6]:
summary_data = []

for t_name in transfer_sets:
    row = {"Transfer Set": t_name}
    for vp in val_paths:
        accs = results[t_name][vp]
        mean_acc = np.mean(accs)
        std_acc = np.std(accs)
        col_name = vp.split("/")[-2] if "/" in vp else vp.split("\\")[-2]
        row[col_name] = f"{mean_acc:.2f} ± {std_acc:.2f}"
    summary_data.append(row)

df_rp = pd.DataFrame(summary_data)
df_rp.to_csv("wear_RP_all_thicknesses_summary.csv", index=False)
print("Consolidated RP (Force) results:")
display(df_rp)

Consolidated RP (Force) results:


,Transfer Set,T185,T188,T191,T194,T197
0,train_20,67.34 ± 5.71,70.63 ± 3.81,84.86 ± 3.38,86.61 ± 3.25,92.82 ± 0.55
1,train_50,73.29 ± 2.87,74.41 ± 3.08,86.52 ± 2.69,89.14 ± 1.97,95.84 ± 0.49
2,train_100,81.23 ± 1.13,80.67 ± 2.29,90.67 ± 4.46,91.48 ± 5.47,98.08 ± 0.70
3,train_200,77.86 ± 5.17,78.20 ± 7.28,91.71 ± 4.39,92.35 ± 4.67,98.73 ± 0.67
4,train_300,77.29 ± 2.23,79.89 ± 3.14,92.37 ± 4.18,94.44 ± 3.34,99.20 ± 0.38
5,train_400,79.79 ± 2.02,81.19 ± 2.67,92.37 ± 4.23,94.97 ± 4.81,99.36 ± 0.26
6,train_500,75.46 ± 1.76,76.77 ± 0.96,92.56 ± 3.03,96.79 ± 1.69,99.46 ± 0.10
